# Goldstein-price function

**Contributed by**: Benoît Legat

In this example, we consider the minimization of the [Goldstein-price function](https://en.wikipedia.org/wiki/Test_functions_for_optimization).

In [1]:
using SumOfSquares
using DynamicPolynomials

Create *symbolic* variables (not JuMP *decision* variables)

In [2]:
@polyvar x[1:2]

(DynamicPolynomials.Variable{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}}[x₁, x₂],)

To use Sum-of-Squares Programming, we first need to pick an SDP solver,
see [here](https://jump.dev/JuMP.jl/stable/installation/#Supported-solvers) for a list of the available choices.

In [3]:
import Clarabel
using Dualization
model = SOSModel(dual_optimizer(Clarabel.Optimizer))

A JuMP Model
├ solver: Dual model with Clarabel attached
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

Create a JuMP decision variable for the lower bound

In [4]:
@variable(model, γ)

γ

`f(x)` is the Goldstein-Price function

In [5]:
f1 = x[1] + x[2] + 1
f2 = 19 - 14*x[1] + 3*x[1]^2 - 14*x[2] + 6*x[1]*x[2] + 3*x[2]^2
f3 = 2*x[1] - 3*x[2]
f4 = 18 - 32*x[1] + 12*x[1]^2 + 48*x[2] - 36*x[1]*x[2] + 27*x[2]^2
f = (1 + f1^2*f2) * (30 + f3^2*f4)

600 + 720x₂ + 720x₁ + 3060x₂² - 4680x₁x₂ + 1260x₁² + 12288x₂³ - 19296x₁x₂² + 7344x₁²x₂ - 1072x₁³ + 14346x₂⁴ - 23616x₁x₂³ + 7776x₁²x₂² + 5784x₁³x₂ - 2454x₁⁴ + 1944x₂⁵ - 11880x₁x₂⁴ + 5040x₁²x₂³ + 9840x₁³x₂² - 7680x₁⁴x₂ + 1344x₁⁵ - 4428x₂⁶ - 1188x₁x₂⁵ + 8730x₁²x₂⁴ + 1240x₁³x₂³ - 5370x₁⁴x₂² - 168x₁⁵x₂ + 952x₁⁶ - 648x₂⁷ + 1944x₁x₂⁶ + 3672x₁²x₂⁵ - 3480x₁³x₂⁴ - 4080x₁⁴x₂³ + 2592x₁⁵x₂² + 1344x₁⁶x₂ - 768x₁⁷ + 729x₂⁸ + 972x₁x₂⁷ - 1458x₁²x₂⁶ - 1836x₁³x₂⁵ + 1305x₁⁴x₂⁴ + 1224x₁⁵x₂³ - 648x₁⁶x₂² - 288x₁⁷x₂ + 144x₁⁸

Constraints `f(x) - γ` to be a sum of squares

In [6]:
con_ref = @constraint(model, f >= γ)
@objective(model, Max, γ)
optimize!(model)

-------------------------------------------------------------
           Clarabel.jl v0.11.1  -  Clever Acronym              

                   (c) Paul Goulart                          
                University of Oxford, 2022                   
-------------------------------------------------------------

problem:
  variables     = 45
  constraints   = 121
  nnz(P)        = 0
  nnz(A)        = 121
  cones (total) = 2
    : Zero        = 1,  numel = 1
    : PSDTriangle = 1,  numel = 120

settings:
  linear algebra: direct / qdldl, precision: 64 bit (1 thread)
  max iter = 200, time limit = Inf,  max step = 0.990
  tol_feas = 1.0e-08, tol_gap_abs = 1.0e-08, tol_gap_rel = 1.0e-08,
  static reg : on, ϵ1 = 1.0e-08, ϵ2 = 4.9e-32
  dynamic reg: on, ϵ = 1.0e-13, δ = 2.0e-07
  iter refine: on, reltol = 1.0e-13, abstol = 1.0e-12, 
               max iter = 10, stop ratio = 5.0
  equilibrate: on, min_scale = 1.0e-04, max_scale = 1.0e+04
               max iter = 10

iter    pcost        dc

The lower bound found is 3

In [7]:
solution_summary(model)

solution_summary(; result = 1, verbose = false)
├ solver_name          : Dual model with Clarabel attached
├ Termination
│ ├ termination_status : ALMOST_OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : ALMOST_SOLVED
├ Solution (result = 1)
│ ├ primal_status        : NEARLY_FEASIBLE_POINT
│ ├ dual_status          : NEARLY_FEASIBLE_POINT
│ ├ objective_value      : 3.00018e+00
│ └ dual_objective_value : 3.00015e+00
└ Work counters
  ├ solve_time (sec)   : 9.99800e-03
  └ barrier_iterations : 17

The moment matrix is as follows, we can already see the global minimizer
`[0, -1]` from the entries `(2, 1)` and `(3, 1)`.
This heuristic way to obtain solutions to the polynomial optimization problem
is suggested in [Laurent2008; (6.15)](@cite).

In [8]:
ν = moment_matrix(con_ref)

MomentMatrix with row/column basis:
 SubBasis{Monomial}([1, x[2], x[1], x[2]^2, x[1]*x[2], x[1]^2, x[2]^3, x[1]*x[2]^2, x[1]^2*x[2], x[1]^3, x[2]^4, x[1]*x[2]^3, x[1]^2*x[2]^2, x[1]^3*x[2], x[1]^4])
And entries in a 15×15 SymMatrix{Float64}:
  1.0000000057710337     -1.0000004273918053     …   7.255092292054487e-6
 -1.0000004273918053      1.0000022281366363         3.0438287918503613e-6
 -2.185488638696298e-5    2.2551280415291303e-5      1.5300578036729643e-5
  1.0000022884832527     -1.0000039022510643         0.00027852799875026694
  2.2539365519936027e-5  -2.1946331012303894e-5      0.00031187389425286886
  2.0673450976409084e-6   4.524301466179731e-7   …   0.000603304126704755
 -1.0000038753463425      1.000005994691415         -0.0006149608786319453
 -2.195538363348391e-5    2.2167992526827632e-5      1.7274101076964203e-5
  4.524114131820914e-7    6.018053324435994e-7      -0.0001409755748564501
  2.854586492280375e-6    1.6778624188613109e-6      0.0010398059287169417
  1.0000

Many entries of the matrix actually have the same moment.
We can obtain the following list of these moments without duplicates
(ignoring when difference of entries representing the same moments is below `1e-5`)

In [9]:
μ = moment_vector(ν, atol = 1e-5)

MomentVector{Float64, StarAlgebras.SubBasis{MultivariateBases.Polynomial{Monomial, Vector{DynamicPolynomials.Variable{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}}}, Vector{Int64}}, Int64, Vector{Int64}, StarAlgebras.MappedBasis{MultivariateBases.Polynomial{Monomial, Vector{DynamicPolynomials.Variable{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}}}, Vector{Int64}}, Vector{Int64}, MultivariatePolynomials.ExponentsIterator{MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}, Nothing, Vector{Int64}}, MultivariateBases.Variables{Monomial, Vector{DynamicPolynomials.Variable{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}}}}, typeof(MultivariatePolynomials.exponents)}, Vector{Vector{Int64}}}, Vector{Float64}}([1.0000000057710337,

The truncated moment matrix can then be obtained as follows

In [10]:
ν_truncated = moment_matrix(μ, monomials(x, 0:3))

MomentMatrix with row/column basis:
 SubBasis{Monomial}([1, x[2], x[1], x[2]^2, x[1]*x[2], x[1]^2, x[2]^3, x[1]*x[2]^2, x[1]^2*x[2], x[1]^3])
And entries in a 10×10 SymMatrix{Float64}:
  1.0000000057710337     -1.0000004273918053     …  2.854586492280375e-6
 -1.0000004273918053      1.0000022884832527        1.6442272241539395e-6
 -2.185488638696298e-5    2.2539365519936027e-5     7.255092292054487e-6
  1.0000022884832527     -1.0000038753463425        6.257459400344528e-7
  2.2539365519936027e-5  -2.195538363348391e-5      3.0438287918503613e-6
  2.0673450976409084e-6   4.524114131820914e-7   …  1.5300578036729643e-5
 -1.0000038753463425      1.0000060012432925        0.00011075463048749017
 -2.195538363348391e-5    2.2205141352578553e-5     0.00027852799875026694
  4.524114131820914e-7    6.198908505267317e-7      0.00031187389425286886
  2.854586492280375e-6    1.6442272241539395e-6     0.000603304126704755

Let's check if the flatness property is satisfied.
The rank of `ν_truncated` seems to be 1:

In [11]:
using LinearAlgebra
LinearAlgebra.svdvals(Matrix(ν_truncated.Q))
LinearAlgebra.rank(Matrix(ν_truncated.Q), rtol = 1e-3)
svdvals(Matrix(ν_truncated.Q))

10-element Vector{Float64}:
 4.000039534511101
 0.0009327548938183214
 0.0001585348291515868
 1.5442435443337745e-5
 1.3783533810691266e-6
 7.392258347263627e-7
 2.2770738898526414e-7
 1.53354513375702e-7
 1.0805118879249214e-7
 7.441721551210851e-9

The rank of `ν` is clearly higher than 1, closer to 3:

In [12]:
svdvals(Matrix(ν.Q))

15-element Vector{Float64}:
 5.088171442490779
 1.563545607728077
 1.0067567473960353
 0.0027161446837601737
 0.0007978811281212177
 0.00016361427994478528
 0.00013045319115424558
 1.8025139792419923e-5
 1.5476282482364761e-6
 6.407070449649256e-7
 1.7492261294408194e-7
 1.588284268692554e-7
 2.366952450403114e-8
 8.374297811292025e-9
 3.8566802486950385e-10

Even if the flatness property is not satisfied, we can
still try extracting the minimizer with a low rank decomposition of rank 3.
We find the optimal solution again doing so:

In [13]:
atomic_measure(ν, FixedRank(3))

Atomic measure on the variables x[1], x[2] with 1 atoms:
 at [-2.1771228628828633e-5, -1.0000013714213587] with weight 1.0206673966292732

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*